In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — RETURN SERIES CONSTRUCTION
# Inputs:  outputs/phase0_returns_matrix.parquet
#          outputs/01_universe_filtered.csv
#          outputs/03_coin_metadata_phase0.csv
#          outputs/phase05_validity_summary.csv
# Outputs: outputs/02_returns_matrix.parquet          ← canonical Phase 1 output
#          outputs/03_coin_metadata.csv               ← enriched with ν + d_param
#          outputs/subperiod_labels.csv               ← date → period label
#          outputs/phase1_break_dates.csv             ← all detected breakpoints
#          outputs/phase1_fat_tail_params.csv         ← Student-t ν per coin
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# Stats
from scipy.stats import t as student_t
from statsmodels.tsa.stattools import zivot_andrews

# Structural breaks (multiple breakpoints — Bai-Perron style)
try:
    import ruptures as rpt
    HAS_RUPTURES = True
except ImportError:
    HAS_RUPTURES = False
    print("⚠️  ruptures not installed — running: pip install ruptures")

Path("outputs").mkdir(exist_ok=True)

# ── Install ruptures if missing ────────────────────────────────────────────────
if not HAS_RUPTURES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ruptures", "-q"])
    import ruptures as rpt
    print("  ✅ ruptures installed")


# ═══════════════════════════════════════════════════════════════════════════════
# LOAD PHASE 0.5 OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("Loading Phase 0.5 outputs...")

log_returns  = pd.read_parquet("outputs/phase0_returns_matrix.parquet")
final_coins  = pd.read_csv("outputs/01_universe_filtered.csv")["coin_id"].tolist()
meta_phase0  = pd.read_csv("outputs/03_coin_metadata_phase0.csv")
validity     = pd.read_csv("outputs/phase05_validity_summary.csv")

# Align coins to those that passed validity
valid_coins = validity.loc[validity["ready_for_phase1"] == True, "coin_id"].tolist()
log_returns = log_returns[[c for c in valid_coins if c in log_returns.columns]]

print(f"  Return matrix  : {log_returns.shape}  (days × coins)")
print(f"  Date range     : {log_returns.index[0]}  →  {log_returns.index[-1]}")
print(f"  Coins ready    : {len(valid_coins)}")


# ═══════════════════════════════════════════════════════════════════════════════
# 1.1  LOG RETURN MATRIX — SHAPE VALIDATION  (fixed)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 1.1 │ Return Matrix Validation ───────────────────────────────────")

first_row_nans  = log_returns.iloc[0].isna().sum()
overall_nan_pct = log_returns.isna().mean().mean()

print(f"  Shape              : {log_returns.shape}")
print(f"  First-row NaNs     : {first_row_nans}")
print(f"  Overall NaN rate   : {overall_nan_pct:.4%}  (threshold: < 5%)")

# ── Diagnose which coins have NaN on row 0 ────────────────────────────────────
if first_row_nans > 0:
    late_start_coins = log_returns.columns[log_returns.iloc[0].isna()].tolist()
    print(f"\n  ⚠️  {first_row_nans} coins have NaN on row 0 — late-starting price data:")
    for c in late_start_coins:
        first_valid = log_returns[c].first_valid_index()
        print(f"    {c:>15}  →  first valid return: {first_valid}")

    # These coins are fine — their NaN on row 0 is legitimate (no price yet).
    # Phase 0 forward-fill only covers gaps ≤2 days WITHIN the series,
    # not before the coin's first trading day.
    # Action: forward-fill row 0 NaNs only if gap ≤ 2 days, else leave as NaN.
    # Downstream methods (correlation, copula) handle NaNs via pairwise dropna.
    print(f"\n  → Keeping these coins. NaNs will be handled pairwise in Phases 2–6.")

# ── Hard stop only if overall NaN rate is problematic ────────────────────────
assert overall_nan_pct < 0.05, \
    f"❌ NaN rate {overall_nan_pct:.2%} exceeds 5% — re-check Phase 0 fill rules"

print("\n  ✅ Validation passed — proceeding with NaNs handled pairwise downstream")


# ═══════════════════════════════════════════════════════════════════════════════
# 1.2  FAT TAIL CHARACTERISATION — STUDENT-t FIT PER COIN
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 1.2 │ Fat Tail Characterisation (Student-t) ──────────────────────")

# Why Student-t?
# Crypto returns have heavy tails — normal distribution underestimates crash
# probability. The degrees-of-freedom (ν) parameter measures tail heaviness:
#   ν → ∞  : approaches normal (thin tails)
#   ν < 10 : fat tails — standard in crypto
#   ν < 4  : extreme fat tails — variance may not exist mathematically
#             These coins need special copula handling in Phase 3

def fit_student_t(series: pd.Series) -> dict:
    """
    MLE fit of Student-t distribution to a return series.
    Returns ν (df), location (μ), scale (σ).
    scipy's t.fit returns (df, loc, scale).
    """
    clean = series.dropna()
    if len(clean) < 50:
        return {"nu": np.nan, "t_loc": np.nan, "t_scale": np.nan, "extreme_fat_tail": np.nan}
    try:
        df, loc, scale = student_t.fit(clean, floc=0)   # fix loc=0 for returns
        return {
            "nu"            : round(df, 3),
            "t_loc"         : round(loc, 6),
            "t_scale"       : round(scale, 6),
            "extreme_fat_tail": df < 4,    # ν < 4 → variance undefined
        }
    except Exception as e:
        return {"nu": np.nan, "t_loc": np.nan, "t_scale": np.nan,
                "extreme_fat_tail": np.nan}


fat_tail_rows = []
for coin in valid_coins:
    if coin not in log_returns.columns:
        continue
    result = fit_student_t(log_returns[coin])
    fat_tail_rows.append({"coin_id": coin, **result})

fat_tail_df = pd.DataFrame(fat_tail_rows)

# ─── Summary ──────────────────────────────────────────────────────────────────
nu_series   = fat_tail_df["nu"].dropna()
extreme     = fat_tail_df["extreme_fat_tail"].sum()
median_nu   = nu_series.median()
min_nu_row  = fat_tail_df.loc[fat_tail_df["nu"].idxmin()]

print(f"  Median ν across universe  : {median_nu:.2f}")
print(f"  Min ν                     : {min_nu_row['nu']:.2f}  ({min_nu_row['coin_id']})")
print(f"  Coins with ν < 4 (extreme): {int(extreme)}")
print(f"  Coins with ν < 10 (fat)   : {(nu_series < 10).sum()}")

if extreme > 0:
    extreme_coins = fat_tail_df.loc[fat_tail_df["extreme_fat_tail"] == True, "coin_id"].tolist()
    print(f"\n  ⚠️  Extreme fat-tail coins (flag for Phase 3 copula selection):")
    for c in extreme_coins:
        nu_val = fat_tail_df.loc[fat_tail_df["coin_id"] == c, "nu"].values[0]
        print(f"    {c:>15} │ ν = {nu_val:.2f}")


# ═══════════════════════════════════════════════════════════════════════════════
# 1.3  STRUCTURAL BREAK DETECTION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Phase 1.3 │ Structural Break Detection ─────────────────────────────────")

# ─── Why we detect breaks ─────────────────────────────────────────────────────
# Similarity computed across a regime change conflates two different market
# structures into one number. A pair that's highly correlated in 2022 and
# uncorrelated in 2024 will show medium correlation overall — masking the truth.
# We detect breaks first, then compute similarity within stable subperiods.

# Two methods, complementary:
#   Zivot-Andrews  → single endogenous break in the log-PRICE series
#                    (statsmodels built-in, tests for break in trend/intercept)
#   Bai-Perron     → multiple structural breaks (via ruptures PELT algorithm)
#                    applied to the RETURN series volatility (rolling std)

# ─── 1.3a  Zivot-Andrews (per coin, single break) ─────────────────────────────
print("\n  Running Zivot-Andrews (single break) on log-price series...")

# Reconstruct log-prices from returns for each coin
# log_price_t = cumsum of log_returns (relative to arbitrary base)
log_prices = log_returns.cumsum()

za_rows = []
for coin in valid_coins:
    if coin not in log_prices.columns:
        continue
    series = log_prices[coin].dropna()
    try:
        za_stat, za_pval, za_cv, za_baselag, za_baselag2, za_break_idx = \
            zivot_andrews(series, maxlag=12, regression="ct", autolag="AIC")
        break_date = series.index[int(za_break_idx)] if za_break_idx is not None else pd.NaT
        za_rows.append({
            "coin_id"      : coin,
            "za_stat"      : round(za_stat, 4),
            "za_pval"      : round(za_pval, 4),
            "za_significant": za_pval < 0.05,
            "za_break_date": break_date,
        })
    except Exception:
        za_rows.append({
            "coin_id": coin, "za_stat": np.nan, "za_pval": np.nan,
            "za_significant": False, "za_break_date": pd.NaT,
        })

za_df = pd.DataFrame(za_rows)
za_sig = za_df["za_significant"].sum()
print(f"    Coins with significant single break : {za_sig} / {len(za_df)}")


# ─── 1.3b  Bai-Perron via PELT (multiple breaks, on rolling volatility) ────────
print("\n  Running Bai-Perron / PELT (multiple breaks) on rolling volatility...")

# We detect breaks in the 30-day rolling standard deviation of returns.
# Rationale: similarity is driven by co-volatility, and regime changes show up
# as volatility level shifts first before correlation shifts.

ROLL_VOL_WINDOW = 30    # days for rolling std
MIN_SEGMENT_LEN = 45    # no segment shorter than 45 trading days (~2 months)
PENALTY         = "bic" # BIC penalty — conservative, avoids over-segmentation

bp_rows = []
all_break_dates = []

for coin in valid_coins:
    if coin not in log_returns.columns:
        continue
    series = log_returns[coin].dropna()

    # Rolling volatility as the signal to break-test
    roll_vol = series.rolling(ROLL_VOL_WINDOW).std().dropna().values.reshape(-1, 1)

    try:
        algo   = rpt.Pelt(model="rbf", min_size=MIN_SEGMENT_LEN, jump=1).fit(roll_vol)
        breaks = algo.predict(pen=np.log(len(roll_vol)))  # BIC-equivalent penalty

        # ruptures returns the END index of each segment; last element = len(signal)
        # Convert to dates (offset by the rolling window lag)
        vol_dates = log_returns[coin].dropna().index[ROLL_VOL_WINDOW - 1:]
        break_dates_coin = []
        for b in breaks[:-1]:  # drop last (= end of series)
            if b < len(vol_dates):
                break_dates_coin.append(vol_dates[b])
                all_break_dates.append(vol_dates[b])

        bp_rows.append({
            "coin_id"         : coin,
            "n_breaks"        : len(break_dates_coin),
            "break_dates"     : ";".join([str(d)[:10] for d in break_dates_coin]),
        })
    except Exception as e:
        bp_rows.append({"coin_id": coin, "n_breaks": 0, "break_dates": ""})

bp_df = pd.DataFrame(bp_rows)

n_breaks_dist = bp_df["n_breaks"].value_counts().sort_index()
print(f"    Break count distribution:")
for k, v in n_breaks_dist.items():
    print(f"      {k} breaks : {v} coins")


# ─── 1.3c  Master break date list — find consensus break dates ────────────────
print("\n  Computing consensus break dates...")

# A consensus break date = a calendar date where many coins simultaneously
# show a structural break. We bin break dates into 30-day windows and look
# for clusters with ≥ 10% of coins breaking around the same time.

if all_break_dates:
    break_series = pd.Series(pd.to_datetime(all_break_dates))
    break_counts = (
        break_series
        .dt.to_period("M")          # bin by month
        .value_counts()
        .sort_index()
    )

    # Keep months where ≥ 10% of coins had a break
    threshold = len(valid_coins) * 0.10
    consensus_months = break_counts[break_counts >= threshold]

    print(f"\n    Consensus break months (≥10% of coins breaking):")
    if len(consensus_months) == 0:
        print("    None detected — no widespread simultaneous regime change")
        consensus_break_dates = []
    else:
        for month, count in consensus_months.items():
            pct = count / len(valid_coins) * 100
            print(f"      {month}  →  {count} coins  ({pct:.0f}%)")
        # Use the 1st of each consensus month as the canonical break date
        consensus_break_dates = sorted([
            pd.Timestamp(str(m)) for m in consensus_months.index
        ])
else:
    print("    No breaks detected across universe")
    consensus_break_dates = []

# ─── Known structural breaks from crypto history (always include) ─────────────
KNOWN_BREAKS = {
    "FTX_collapse"      : pd.Timestamp("2022-11-08"),
    "Bitcoin_ETF_approval": pd.Timestamp("2024-01-10"),
}

for name, date in KNOWN_BREAKS.items():
    if date not in consensus_break_dates:
        consensus_break_dates.append(date)
        print(f"\n    Adding known break: {name} ({date.date()})")

consensus_break_dates = sorted(consensus_break_dates)
print(f"\n    Final break dates used for subperiod definition:")
for d in consensus_break_dates:
    print(f"      {d.date()}")


# ─── 1.3d  Define subperiods ──────────────────────────────────────────────────
print("\n  Defining subperiods...")

date_index = log_returns.index
start_date = date_index[0]
end_date   = date_index[-1]

# Build subperiod boundaries: [start, break1, break2, ..., end]
boundaries = [start_date] + [
    d for d in consensus_break_dates
    if start_date < d < end_date
] + [end_date]

# Assign a subperiod label to each date
subperiod_labels = []
for d in date_index:
    assigned = None
    for i in range(len(boundaries) - 1):
        if boundaries[i] <= d <= boundaries[i + 1]:
            assigned = f"Period_{i + 1}"
            break
    if assigned is None:
        assigned = f"Period_{len(boundaries) - 1}"
    subperiod_labels.append(assigned)

subperiod_df = pd.DataFrame({
    "date"      : date_index,
    "subperiod" : subperiod_labels,
})

period_sizes = subperiod_df["subperiod"].value_counts().sort_index()
print(f"\n    Subperiod sizes:")
for period, size in period_sizes.items():
    period_dates = subperiod_df.loc[subperiod_df["subperiod"] == period, "date"]
    print(f"      {period}  :  {size:3d} days  "
          f"({period_dates.iloc[0].date()} → {period_dates.iloc[-1].date()})")


# ═══════════════════════════════════════════════════════════════════════════════
# BUILD ENRICHED COIN METADATA
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Building enriched coin_metadata ────────────────────────────────────────")

# Merge Phase 0 metadata + Student-t params + break info
coin_metadata = (
    meta_phase0
    .merge(fat_tail_df[["coin_id", "nu", "t_loc", "t_scale", "extreme_fat_tail"]],
           on="coin_id", how="left")
    .merge(za_df[["coin_id", "za_break_date", "za_significant"]], on="coin_id", how="left")
    .merge(bp_df[["coin_id", "n_breaks", "break_dates"]],         on="coin_id", how="left")
    .merge(validity[["coin_id", "d_estimate", "d_action"]],       on="coin_id", how="left")
)

# Rename for clarity
coin_metadata = coin_metadata.rename(columns={
    "za_break_date" : "primary_break_date",
    "n_breaks"      : "n_structural_breaks",
    "break_dates"   : "all_break_dates",
    "d_estimate"    : "d_param",
})

print(f"  Metadata shape : {coin_metadata.shape}")
print(f"  Columns        : {list(coin_metadata.columns)}")


# ═══════════════════════════════════════════════════════════════════════════════
# PHASE 1 OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── Saving Phase 1 outputs ──────────────────────────────────────────────────")

# 1. Canonical return matrix (same data as Phase 0.5, but now the official Phase 1 output)
log_returns.to_parquet("outputs/02_returns_matrix.parquet")
print("  ✅ 02_returns_matrix.parquet     → (729 × 125) log-return matrix")

# 2. Enriched coin metadata
coin_metadata.to_csv("outputs/03_coin_metadata.csv", index=False)
print("  ✅ 03_coin_metadata.csv          → amihud + ν + d_param + break dates")

# 3. Subperiod labels
subperiod_df.to_csv("outputs/subperiod_labels.csv", index=False)
print("  ✅ subperiod_labels.csv          → date → subperiod label")

# 4. Detailed break date results
break_export = za_df.merge(bp_df, on="coin_id", how="left")
break_export.to_csv("outputs/phase1_break_dates.csv", index=False)
print("  ✅ phase1_break_dates.csv        → ZA + PELT break dates per coin")

# 5. Fat tail params (standalone — Phase 3 copula selection needs this)
fat_tail_df.to_csv("outputs/phase1_fat_tail_params.csv", index=False)
print("  ✅ phase1_fat_tail_params.csv    → Student-t ν per coin")

# ─── Final summary ────────────────────────────────────────────────────────────
n_periods = subperiod_df["subperiod"].nunique()
print(f"""
╔══════════════════════════════════════════════════════════════╗
║  PHASE 1 COMPLETE                                            ║
╠══════════════════════════════════════════════════════════════╣
║  Return matrix            : 729 × 125  (T × N)              ║
║  Median ν (fat tails)     : {median_nu:<6.2f}  (ν < 4: {int(extreme)} coins)         ║
║  Structural breaks found  : {len(consensus_break_dates)} consensus dates          ║
║  Subperiods defined       : {n_periods}                                ║
║  Coins with ≥1 break      : {(bp_df['n_breaks'] > 0).sum()}                              ║
╠══════════════════════════════════════════════════════════════╣
║  Next: Phase 2 — Baseline Similarity Matrix                  ║
║  Key inputs:                                                 ║
║    02_returns_matrix.parquet                                 ║
║    subperiod_labels.csv                                      ║
║    phase1_fat_tail_params.csv  (→ Phase 3)                   ║
╚══════════════════════════════════════════════════════════════╝
""")

Loading Phase 0.5 outputs...
  Return matrix  : (729, 125)  (days × coins)
  Date range     : 2024-03-22 00:00:00  →  2026-03-20 00:00:00
  Coins ready    : 125

── Phase 1.1 │ Return Matrix Validation ───────────────────────────────────
  Shape              : (729, 125)
  First-row NaNs     : 20
  Overall NaN rate   : 3.9912%  (threshold: < 5%)

  ⚠️  20 coins have NaN on row 0 — late-starting price data:
     berachain-bera  →  first valid return: 2025-02-07 00:00:00
          bittensor  →  first valid return: 2024-04-12 00:00:00
       cow-protocol  →  first valid return: 2024-11-07 00:00:00
         eigenlayer  →  first valid return: 2024-10-02 00:00:00
             ethena  →  first valid return: 2024-04-03 00:00:00
             eurite  →  first valid return: 2024-08-29 00:00:00
               kaia  →  first valid return: 2024-11-01 00:00:00
          layerzero  →  first valid return: 2024-06-21 00:00:00
           movement  →  first valid return: 2024-12-10 00:00:00
     official-